# C05固定三网络扩展
新增Music-Rev、NDC-classes、high-school_2012；只比较筛选阶数1/10，每组5重复，不跑S2V。
保留全部结果及原Algebra/HCL观察。本机后台已启动时不要再次启动此Notebook。
命令行与Notebook二选一，网络名单预先固定，不根据效果选取。

In [ ]:
from pathlib import Path
import os,sys,subprocess,signal
import torch
ROOT=Path.cwd().resolve()
PLAN=ROOT/'plan_order_extension.json'
DATA_ROOT=Path('D:/YX/HyperGraphMatrix')
CACHE_ROOT=Path('D:/YX/Hyper_Distance')
OUTPUT=ROOT/'results/extension'
# 服务器数据路径按实际位置改，使用之前已安装torch的CNN内核。
WORKERS=3
if os.name=='nt':os.environ['PYTHONPATH']=str(ROOT/'.ops')+os.pathsep+os.environ.get('PYTHONPATH','')
def run_cli(*args):
    cmd=[sys.executable,'-u',str(ROOT/'run_suite.py'),*map(str,args)]
    process=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=(os.name=='posix'))
    try:
        for line in process.stdout:print(line,end='',flush=True)
        status=process.wait()
        if status:raise subprocess.CalledProcessError(status,cmd)
    except BaseException:
        if process.poll() is None:
            if os.name=='posix':os.killpg(process.pid,signal.SIGINT)
            else:process.terminate()
        raise
print(sys.executable)

## 检查数据及缓存

In [ ]:
run_cli('check','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT)

## 按固定网络顺序运行，每个完成即汇总
每个网络10次训练。完成项会跳过，中断训练接续最近完整回合。

In [ ]:
for network in ['Music-Rev','NDC-classes','high-school_2012']:
    print('开始网络:',network)
    run_cli('run','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT,
            '--task-file',ROOT/f'tasks_{network}.json','--output',OUTPUT,'--workers',WORKERS)
    result=subprocess.run([sys.executable,str(ROOT/'summarize_extension.py'),'--allow-partial'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print(result.stdout)
    result.check_returncode()


## 查看汇总

In [ ]:
print((ROOT/'comparison/扩展结果.md').read_text(encoding='utf-8'))